# Lumen AI — BAU vs Optimizer Evaluation Plots

This Databricks notebook is designed to be **followable and editable**, not a black-box utility.

It assumes one row per customer with:

- Historical BAU channel assignment columns, e.g. `bau_leads`, `bau_email`, ...
- Model-predicted lift / score columns per channel, e.g. `lift_leads`, `lift_email`, ...
- Optimizer decision columns for two scenarios, e.g. `opt1_leads`, `opt2_leads`, ...
- Historical conversion label, e.g. `converted`
- PRISM any-credit-card propensity, e.g. `prism_any_cc_propensity`

Main outputs:

1. **Plot 1:** Channel distribution by PRISM propensity decile for BAU vs scenario 1 vs scenario 2
2. **Plot 2:** Match-group validation — when BAU and optimizer agree, do actual converters concentrate in high predicted-lift deciles?
3. **Plot 3A:** Non-match raw lift comparison — predicted BAU lift vs optimizer lift
4. **Plot 3B:** Non-match expected conversion improvement — actual BAU conversion + model-estimated incremental lift
5. **Plot 3C:** Transparent calculation table for the expected improvement number

No Excel export is included. The notebook displays tables and plots directly.

## Cell 1 — Imports and display setup

This cell imports the Python libraries used throughout the notebook and sets Pandas display options so the summary tables are easier to inspect in Databricks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## Cell 2 — Configuration

Edit this cell first. Most future changes should happen here:

- `CHANNELS`: channel suffixes used in your column names
- `BAU_PREFIX`: prefix for historical BAU binary treatment columns
- `LIFT_PREFIX`: prefix for model-predicted lift columns
- `SCENARIOS`: optimizer scenario names and prefixes
- `CONVERSION_COL`: historical conversion label
- `PRISM_PROPENSITY_COL`: PRISM any-credit-card propensity column

The notebook expects columns like `bau_leads`, `lift_leads`, `opt1_leads`, and `opt2_leads` for each channel.

In [ ]:
CHANNELS = [
    "leads",
    "email",
    "direct_mail",
    "phone",
    "mobile",
    "branch",
    "web",
]

# Replace with your real table name if loading from a Spark table.
INPUT_TABLE = "your_catalog.your_schema.customer_optimizer_outputs"

CUSTOMER_ID_COL = "customer_id"
CONVERSION_COL = "converted"
PRISM_PROPENSITY_COL = "prism_any_cc_propensity"

BAU_PREFIX = "bau_"
LIFT_PREFIX = "lift_"

SCENARIOS = {
    "Scenario 1 — Unconstrained": "opt1_",
    "Scenario 2 — Leads cap 30K": "opt2_",
}

# Decile convention: 1 = highest score / highest lift, 10 = lowest.
N_DECILES = 10

## Cell 3 — Load your data

Use one of the loading options below. The output should be a Pandas DataFrame called `df_raw`.

For large tables, consider filtering to the evaluation population in Spark first before converting to Pandas.

In [ ]:
# Option A — Load from Spark table
# spark_df = spark.table(INPUT_TABLE)
# df_raw = spark_df.toPandas()

# Option B — Use an existing Spark DataFrame from earlier notebook cells
# df_raw = your_spark_dataframe.toPandas()

# Option C — Use a Pandas DataFrame already created in this notebook
# df_raw = your_pandas_dataframe.copy()

# Safety check: uncomment after loading
# print(df_raw.shape)
# display(df_raw.head())

## Cell 4 — Optional synthetic data for testing

Run this cell only if you want to test the notebook before connecting your real table. It creates a small fake dataset with the same expected schema.

In [ ]:
CREATE_SYNTHETIC_DATA = False

if CREATE_SYNTHETIC_DATA:
    rng = np.random.default_rng(42)
    n = 50_000
    df_raw = pd.DataFrame({
        CUSTOMER_ID_COL: np.arange(n),
        PRISM_PROPENSITY_COL: rng.beta(2, 8, size=n),
    })
    
    # BAU assignment: more leads for high PRISM customers, otherwise mixed channels.
    prism_rank = df_raw[PRISM_PROPENSITY_COL].rank(pct=True)
    bau_probs = np.vstack([
        0.02 + 0.25 * prism_rank,             # leads
        np.full(n, 0.25),                     # email
        np.full(n, 0.18),                     # direct_mail
        np.full(n, 0.12),                     # phone
        np.full(n, 0.17),                     # mobile
        np.full(n, 0.11),                     # branch
        np.full(n, 0.15),                     # web
    ]).T
    bau_probs = bau_probs / bau_probs.sum(axis=1, keepdims=True)
    bau_idx = np.array([rng.choice(len(CHANNELS), p=p) for p in bau_probs])
    
    # Lift columns: correlated with PRISM + channel-specific patterns.
    base = 0.005 + 0.06 * df_raw[PRISM_PROPENSITY_COL].values
    channel_effect = {
        "leads": 0.018,
        "email": 0.006,
        "direct_mail": 0.004,
        "phone": 0.008,
        "mobile": 0.010,
        "branch": 0.011,
        "web": 0.007,
    }
    for ch in CHANNELS:
        noise = rng.normal(0, 0.008, size=n)
        df_raw[f"{LIFT_PREFIX}{ch}"] = np.clip(base + channel_effect[ch] + noise, -0.02, 0.20)
    
    # Optimizer scenario 1: choose max lift.
    lift_matrix = df_raw[[f"{LIFT_PREFIX}{ch}" for ch in CHANNELS]].to_numpy()
    opt1_idx = lift_matrix.argmax(axis=1)
    
    # Optimizer scenario 2: cap leads at roughly 30K; here translated proportionally for synthetic n.
    opt2_idx = opt1_idx.copy()
    leads_idx = CHANNELS.index("leads")
    lead_candidates = np.where(opt2_idx == leads_idx)[0]
    synthetic_cap = min(30_000, int(0.12 * n))
    if len(lead_candidates) > synthetic_cap:
        lead_lifts = lift_matrix[lead_candidates, leads_idx]
        keep_leads = lead_candidates[np.argsort(-lead_lifts)[:synthetic_cap]]
        drop_leads = np.setdiff1d(lead_candidates, keep_leads)
        lift_without_leads = lift_matrix[drop_leads].copy()
        lift_without_leads[:, leads_idx] = -np.inf
        opt2_idx[drop_leads] = lift_without_leads.argmax(axis=1)
    
    for j, ch in enumerate(CHANNELS):
        df_raw[f"{BAU_PREFIX}{ch}"] = (bau_idx == j).astype(int)
        df_raw[f"opt1_{ch}"] = (opt1_idx == j).astype(int)
        df_raw[f"opt2_{ch}"] = (opt2_idx == j).astype(int)
    
    # Observed conversion under BAU: tied to PRISM and BAU channel lift.
    bau_lift = lift_matrix[np.arange(n), bau_idx]
    p_conv = np.clip(0.015 + 0.50 * bau_lift + 0.04 * df_raw[PRISM_PROPENSITY_COL].values, 0, 0.50)
    df_raw[CONVERSION_COL] = rng.binomial(1, p_conv)
    
    print(df_raw.shape)
    display(df_raw.head())

## Cell 5 — Validation helpers

These helpers check that required columns exist and convert one-hot channel columns into a single selected channel per customer.

Important assumption: for each policy, each customer should usually have exactly one selected channel. If a row has multiple selected channels, this notebook uses the first channel in `CHANNELS`. If a row has no selected channel, the assigned channel becomes missing.

In [ ]:
def required_columns_for_config():
    cols = [CUSTOMER_ID_COL, CONVERSION_COL, PRISM_PROPENSITY_COL]
    cols += [f"{BAU_PREFIX}{ch}" for ch in CHANNELS]
    cols += [f"{LIFT_PREFIX}{ch}" for ch in CHANNELS]
    for prefix in SCENARIOS.values():
        cols += [f"{prefix}{ch}" for ch in CHANNELS]
    return cols


def validate_input_columns(df: pd.DataFrame) -> None:
    missing = [c for c in required_columns_for_config() if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    print("All required columns are present.")


def one_hot_to_channel(df: pd.DataFrame, prefix: str, output_col: str) -> pd.Series:
    channel_cols = [f"{prefix}{ch}" for ch in CHANNELS]
    values = df[channel_cols].fillna(0).to_numpy()
    has_any = values.sum(axis=1) > 0
    chosen_idx = values.argmax(axis=1)
    chosen = np.array(CHANNELS, dtype=object)[chosen_idx]
    chosen = pd.Series(chosen, index=df.index, name=output_col)
    chosen.loc[~has_any] = np.nan
    return chosen


def selected_lift(df: pd.DataFrame, selected_channel_col: str, output_col: str) -> pd.Series:
    out = pd.Series(np.nan, index=df.index, name=output_col, dtype="float")
    for ch in CHANNELS:
        mask = df[selected_channel_col] == ch
        out.loc[mask] = df.loc[mask, f"{LIFT_PREFIX}{ch}"]
    return out


def make_high_to_low_decile(score: pd.Series, n_deciles: int = 10) -> pd.Series:
    """Returns deciles where 1 is highest score and n_deciles is lowest."""
    ranked = score.rank(method="first", ascending=False)
    return pd.qcut(ranked, q=n_deciles, labels=list(range(1, n_deciles + 1))).astype(int)

## Cell 6 — Prepare analysis DataFrame

This cell creates the columns used by all plots:

- `bau_channel`
- `bau_lift`
- `prism_decile`, where 1 is highest PRISM propensity
- For each scenario:
  - optimizer channel
  - optimizer lift
  - match flag
  - lift difference vs BAU

After this cell, most analysis uses `df_eval` rather than `df_raw`.

In [ ]:
validate_input_columns(df_raw)

df_eval = df_raw.copy()

# Normalize conversion label to numeric 0/1.
df_eval[CONVERSION_COL] = df_eval[CONVERSION_COL].astype(float)

# BAU selected channel and BAU lift.
df_eval["bau_channel"] = one_hot_to_channel(df_eval, BAU_PREFIX, "bau_channel")
df_eval["bau_lift"] = selected_lift(df_eval, "bau_channel", "bau_lift")

# PRISM propensity decile: 1 = highest propensity, 10 = lowest.
df_eval["prism_decile"] = make_high_to_low_decile(df_eval[PRISM_PROPENSITY_COL], N_DECILES)

# Scenario-specific selected channels and lifts.
for scenario_name, prefix in SCENARIOS.items():
    safe_name = scenario_name.lower().replace(" ", "_").replace("—", "").replace("-", "_")
    channel_col = f"{safe_name}_channel"
    lift_col = f"{safe_name}_lift"
    match_col = f"{safe_name}_matches_bau"
    delta_col = f"{safe_name}_delta_lift_vs_bau"
    
    df_eval[channel_col] = one_hot_to_channel(df_eval, prefix, channel_col)
    df_eval[lift_col] = selected_lift(df_eval, channel_col, lift_col)
    df_eval[match_col] = df_eval[channel_col].eq(df_eval["bau_channel"])
    df_eval[delta_col] = df_eval[lift_col] - df_eval["bau_lift"]

print(df_eval.shape)
display(df_eval.head())

## Cell 7 — High-level scenario summary table

This table gives the first sanity check:

- What percentage of customers match BAU?
- What is actual historical conversion in match vs non-match groups?
- How much higher is the optimizer-selected lift than BAU on average?

This table is not a final executive plot, but it helps debug whether the results make sense before plotting.

In [ ]:
def scenario_safe_name(scenario_name: str) -> str:
    return scenario_name.lower().replace(" ", "_").replace("—", "").replace("-", "_")

summary_rows = []
for scenario_name in SCENARIOS:
    safe_name = scenario_safe_name(scenario_name)
    match_col = f"{safe_name}_matches_bau"
    lift_col = f"{safe_name}_lift"
    delta_col = f"{safe_name}_delta_lift_vs_bau"
    channel_col = f"{safe_name}_channel"
    
    for group_name, mask in {
        "all": pd.Series(True, index=df_eval.index),
        "match": df_eval[match_col],
        "non_match": ~df_eval[match_col],
    }.items():
        d = df_eval.loc[mask].copy()
        summary_rows.append({
            "scenario": scenario_name,
            "group": group_name,
            "n_customers": len(d),
            "customer_share": len(d) / len(df_eval),
            "actual_conversion_rate": d[CONVERSION_COL].mean(),
            "avg_bau_lift": d["bau_lift"].mean(),
            "avg_optimizer_lift": d[lift_col].mean(),
            "avg_delta_lift_vs_bau": d[delta_col].mean(),
            "expected_incremental_conversions": d[delta_col].sum(),
        })

scenario_summary = pd.DataFrame(summary_rows)
display(scenario_summary)

# Plot 1 — Channel distribution by PRISM propensity decile

Business question:

> For high-PRISM vs low-PRISM customers, how does the channel mix differ across BAU, unconstrained optimization, and constrained optimization?

Interpretation:

- Decile 1 = highest PRISM any-credit-card propensity
- Decile 10 = lowest PRISM propensity
- Each bar sums to 100% within a decile
- A visible shift from BAU to optimizer means Lumen AI is changing the channel allocation logic, not simply reproducing historical rules

## Cell 8 — Prepare Plot 1 table

This creates one long table with policy, PRISM decile, channel, count, and percentage. It is useful to inspect this table before plotting.

In [ ]:
def channel_distribution_by_decile(df: pd.DataFrame, channel_col: str, policy_name: str) -> pd.DataFrame:
    tmp = (
        df.groupby(["prism_decile", channel_col], dropna=False)
          .size()
          .reset_index(name="n_customers")
          .rename(columns={channel_col: "channel"})
    )
    tmp["policy"] = policy_name
    tmp["decile_total"] = tmp.groupby("prism_decile")["n_customers"].transform("sum")
    tmp["pct_customers"] = tmp["n_customers"] / tmp["decile_total"]
    return tmp[["policy", "prism_decile", "channel", "n_customers", "decile_total", "pct_customers"]]

plot1_parts = [channel_distribution_by_decile(df_eval, "bau_channel", "BAU")]
for scenario_name in SCENARIOS:
    safe_name = scenario_safe_name(scenario_name)
    plot1_parts.append(channel_distribution_by_decile(df_eval, f"{safe_name}_channel", scenario_name))

plot1_table = pd.concat(plot1_parts, ignore_index=True)
display(plot1_table.sort_values(["policy", "prism_decile", "channel"]))

## Cell 9 — Plot 1: one stacked bar chart per policy

This produces separate charts for BAU, scenario 1, and scenario 2 to avoid clutter.

Each chart answers: within each PRISM decile, what share of customers gets each channel?

In [ ]:
def plot_channel_mix_for_policy(plot_table: pd.DataFrame, policy_name: str):
    d = plot_table[plot_table["policy"] == policy_name]
    pivot = (
        d.pivot_table(index="prism_decile", columns="channel", values="pct_customers", aggfunc="sum")
         .reindex(index=range(1, N_DECILES + 1), columns=CHANNELS)
         .fillna(0)
    )
    ax = pivot.plot(kind="bar", stacked=True, figsize=(11, 5))
    ax.set_title(f"Channel Distribution by PRISM Decile — {policy_name}")
    ax.set_xlabel("PRISM propensity decile, 1 = highest")
    ax.set_ylabel("% of customers")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    ax.legend(title="Channel", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

for policy in ["BAU"] + list(SCENARIOS.keys()):
    plot_channel_mix_for_policy(plot1_table, policy)

# Plot 2 — Match-group decile validation

Business question:

> When BAU and the optimizer agree on the same channel, does the model rank actual converters toward the highest predicted-lift deciles?

Why this group matters:

- In the match group, the optimizer's chosen channel is the same as the actual historical BAU channel.
- Therefore, the observed conversion happened under the same channel that the model would have selected.
- This gives a partial offline validation of the model ranking.

Interpretation:

- Lift decile 1 = highest optimizer predicted lift
- If actual conversion rate is higher in top lift deciles, the score is directionally useful for ranking customers.

## Cell 10 — Prepare Plot 2 tables

This creates a decile table for each scenario using only customers where optimizer channel = BAU channel.

In [ ]:
def prepare_match_decile_table(df: pd.DataFrame, scenario_name: str) -> pd.DataFrame:
    safe_name = scenario_safe_name(scenario_name)
    match_col = f"{safe_name}_matches_bau"
    lift_col = f"{safe_name}_lift"
    
    d = df.loc[df[match_col]].copy()
    d["lift_decile"] = make_high_to_low_decile(d[lift_col], N_DECILES)
    
    out = (
        d.groupby("lift_decile")
         .agg(
             n_customers=(CUSTOMER_ID_COL, "count"),
             actual_conversions=(CONVERSION_COL, "sum"),
             actual_conversion_rate=(CONVERSION_COL, "mean"),
             avg_optimizer_lift=(lift_col, "mean"),
         )
         .reset_index()
    )
    out["scenario"] = scenario_name
    return out[["scenario", "lift_decile", "n_customers", "actual_conversions", "actual_conversion_rate", "avg_optimizer_lift"]]

plot2_tables = {scenario_name: prepare_match_decile_table(df_eval, scenario_name) for scenario_name in SCENARIOS}
for scenario_name, table in plot2_tables.items():
    print(scenario_name)
    display(table)

## Cell 11 — Plot 2: conversion rate by lift decile

This plot shows actual historical conversion rate for matched customers after ranking them by predicted optimizer lift.

A strong result would show higher conversion in decile 1 than in lower deciles.

In [ ]:
def plot_match_decile_validation(match_table: pd.DataFrame, scenario_name: str):
    d = match_table.sort_values("lift_decile")
    ax = d.plot(x="lift_decile", y="actual_conversion_rate", kind="bar", legend=False, figsize=(9, 4))
    ax.set_title(f"Match-Group Validation — {scenario_name}")
    ax.set_xlabel("Predicted lift decile, 1 = highest")
    ax.set_ylabel("Actual conversion rate")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(f"{h:.1%}", (p.get_x() + p.get_width() / 2, h), ha="center", va="bottom", fontsize=8, rotation=0)
    plt.tight_layout()
    plt.show()

for scenario_name, table in plot2_tables.items():
    plot_match_decile_validation(table, scenario_name)

# Plot 3 — Non-match evaluation family

For non-match customers, BAU and optimizer recommend different channels.

This is the most important but also the most counterfactual part:

- Actual conversion is observed under BAU.
- The optimizer channel was not actually assigned historically.
- Therefore, improvement is model-estimated, not experimentally proven.

We split this into three pieces:

1. **Plot 3A:** Raw model lift comparison
2. **Plot 3B:** Expected conversion improvement bar chart
3. **Plot 3C:** Calculation table explaining the business-impact number

## Cell 12 — Prepare non-match tables

For each scenario, this creates a customer-level non-match table with:

- BAU channel
- optimizer channel
- actual conversion under BAU
- predicted BAU lift
- predicted optimizer lift
- lift difference

This table is useful for debugging and inspecting individual examples.

In [ ]:
def prepare_nonmatch_customer_table(df: pd.DataFrame, scenario_name: str) -> pd.DataFrame:
    safe_name = scenario_safe_name(scenario_name)
    channel_col = f"{safe_name}_channel"
    lift_col = f"{safe_name}_lift"
    match_col = f"{safe_name}_matches_bau"
    delta_col = f"{safe_name}_delta_lift_vs_bau"
    
    cols = [
        CUSTOMER_ID_COL,
        CONVERSION_COL,
        PRISM_PROPENSITY_COL,
        "prism_decile",
        "bau_channel",
        channel_col,
        "bau_lift",
        lift_col,
        delta_col,
    ]
    d = df.loc[~df[match_col], cols].copy()
    d = d.rename(columns={
        channel_col: "optimizer_channel",
        lift_col: "optimizer_lift",
        delta_col: "delta_lift_vs_bau",
    })
    d["scenario"] = scenario_name
    return d

nonmatch_tables = {scenario_name: prepare_nonmatch_customer_table(df_eval, scenario_name) for scenario_name in SCENARIOS}

for scenario_name, table in nonmatch_tables.items():
    print(scenario_name, table.shape)
    display(table.head(20))

## Cell 13 — Plot 3A: raw lift comparison

This is the pure model-score story.

It compares average predicted lift under:

- actual BAU channel
- optimizer-recommended channel

for customers where BAU and optimizer disagree.

This supports the message:

> The optimizer recommends a different channel because the model predicts higher lift for that alternative channel.

In [ ]:
def plot_nonmatch_raw_lift_comparison(nonmatch_table: pd.DataFrame, scenario_name: str):
    avg_bau_lift = nonmatch_table["bau_lift"].mean()
    avg_opt_lift = nonmatch_table["optimizer_lift"].mean()
    avg_delta = nonmatch_table["delta_lift_vs_bau"].mean()
    
    plot_df = pd.DataFrame({
        "policy": ["BAU channel", "Optimizer channel"],
        "avg_predicted_lift": [avg_bau_lift, avg_opt_lift],
    })
    
    ax = plot_df.plot(x="policy", y="avg_predicted_lift", kind="bar", legend=False, figsize=(7, 4))
    ax.set_title(f"Non-Match Raw Lift Comparison — {scenario_name}")
    ax.set_xlabel("")
    ax.set_ylabel("Average predicted lift")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.2%}")
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(f"{h:.2%}", (p.get_x() + p.get_width() / 2, h), ha="center", va="bottom", fontsize=9)
    ax.text(0.5, max(avg_bau_lift, avg_opt_lift) * 1.05, f"Avg lift difference: {avg_delta:+.2%}", ha="center")
    plt.tight_layout()
    plt.show()

for scenario_name, table in nonmatch_tables.items():
    plot_nonmatch_raw_lift_comparison(table, scenario_name)

## Cell 14 — Optional Plot 3A companion: distribution of customer-level lift differences

This histogram shows whether the average improvement is broad-based or driven by a small number of customers.

X-axis:

`optimizer predicted lift - BAU predicted lift`

If most values are above zero, the optimizer improvement is broad-based.

In [ ]:
def plot_delta_lift_distribution(nonmatch_table: pd.DataFrame, scenario_name: str):
    ax = nonmatch_table["delta_lift_vs_bau"].plot(kind="hist", bins=50, figsize=(9, 4))
    ax.set_title(f"Distribution of Predicted Lift Difference — {scenario_name}")
    ax.set_xlabel("Optimizer lift - BAU lift")
    ax.set_ylabel("Number of customers")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")
    plt.tight_layout()
    plt.show()

for scenario_name, table in nonmatch_tables.items():
    plot_delta_lift_distribution(table, scenario_name)

## Cell 15 — Plot 3B/3C calculation function

This cell defines the expected conversion improvement calculation.

For non-match customers:

- Observed BAU conversion rate:

`actual_bau_cr = mean(actual converted)`

- Average model-estimated incremental lift:

`avg_incremental_lift = mean(optimizer_lift - bau_lift)`

- Expected optimizer conversion rate:

`expected_optimizer_cr = actual_bau_cr + avg_incremental_lift`

- Relative improvement:

`relative_lift = (expected_optimizer_cr - actual_bau_cr) / actual_bau_cr`

Important framing:

This is **not** a randomized A/B-test result. It is a model-based counterfactual estimate anchored on actual BAU conversion.

In [ ]:
def expected_conversion_improvement(nonmatch_table: pd.DataFrame, scenario_name: str) -> pd.DataFrame:
    n = len(nonmatch_table)
    actual_bau_cr = nonmatch_table[CONVERSION_COL].mean()
    avg_bau_lift = nonmatch_table["bau_lift"].mean()
    avg_optimizer_lift = nonmatch_table["optimizer_lift"].mean()
    avg_incremental_lift = nonmatch_table["delta_lift_vs_bau"].mean()
    expected_optimizer_cr = actual_bau_cr + avg_incremental_lift
    relative_improvement = np.nan
    if actual_bau_cr != 0 and not pd.isna(actual_bau_cr):
        relative_improvement = (expected_optimizer_cr - actual_bau_cr) / actual_bau_cr
    expected_incremental_conversions = nonmatch_table["delta_lift_vs_bau"].sum()
    observed_conversions = nonmatch_table[CONVERSION_COL].sum()
    expected_optimizer_conversions = observed_conversions + expected_incremental_conversions
    
    return pd.DataFrame([{
        "scenario": scenario_name,
        "n_nonmatch_customers": n,
        "observed_bau_conversions": observed_conversions,
        "actual_bau_conversion_rate": actual_bau_cr,
        "avg_bau_predicted_lift": avg_bau_lift,
        "avg_optimizer_predicted_lift": avg_optimizer_lift,
        "avg_incremental_lift_pp": avg_incremental_lift,
        "expected_optimizer_conversion_rate": expected_optimizer_cr,
        "relative_improvement_vs_bau": relative_improvement,
        "expected_incremental_conversions": expected_incremental_conversions,
        "expected_optimizer_conversions": expected_optimizer_conversions,
    }])

plot3_calc_tables = {scenario_name: expected_conversion_improvement(table, scenario_name) for scenario_name, table in nonmatch_tables.items()}
plot3_calc_summary = pd.concat(plot3_calc_tables.values(), ignore_index=True)
display(plot3_calc_summary)

## Cell 16 — Plot 3B: expected conversion improvement bars

This is the business-impact plot.

It shows:

- Actual BAU conversion rate observed in history
- Expected optimizer conversion rate after adding model-estimated incremental lift

Suggested executive wording:

> Among customers where Lumen AI disagrees with BAU, actual BAU conversion was X%. Based on model-estimated incremental lift, the optimizer allocation is expected to increase conversion to Y%, a Z% relative improvement.

In [ ]:
def plot_expected_conversion_bars(calc_table: pd.DataFrame, scenario_name: str):
    row = calc_table.iloc[0]
    plot_df = pd.DataFrame({
        "policy": ["Actual BAU", "Expected Optimizer"],
        "conversion_rate": [row["actual_bau_conversion_rate"], row["expected_optimizer_conversion_rate"]],
    })
    
    ax = plot_df.plot(x="policy", y="conversion_rate", kind="bar", legend=False, figsize=(7, 4))
    ax.set_title(f"Expected Conversion Improvement — {scenario_name}")
    ax.set_xlabel("")
    ax.set_ylabel("Conversion rate")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(f"{h:.2%}", (p.get_x() + p.get_width() / 2, h), ha="center", va="bottom", fontsize=9)
    
    rel = row["relative_improvement_vs_bau"]
    incr = row["avg_incremental_lift_pp"]
    top = max(plot_df["conversion_rate"].max(), 0.0001)
    ax.text(
        0.5,
        top * 1.08,
        f"Incremental lift: {incr:+.2%} | Relative improvement: {rel:+.1%}",
        ha="center",
    )
    plt.tight_layout()
    plt.show()

for scenario_name, calc_table in plot3_calc_tables.items():
    plot_expected_conversion_bars(calc_table, scenario_name)

## Cell 17 — Plot 3C: transparent calculation table

This is the table you can paste into a slide or use as speaking notes.

It makes the one-number business-impact calculation explicit and auditable.

In [ ]:
def format_plot3_calculation_table(calc_summary: pd.DataFrame) -> pd.DataFrame:
    out = calc_summary.copy()
    rate_cols = [
        "actual_bau_conversion_rate",
        "avg_bau_predicted_lift",
        "avg_optimizer_predicted_lift",
        "avg_incremental_lift_pp",
        "expected_optimizer_conversion_rate",
        "relative_improvement_vs_bau",
    ]
    for c in rate_cols:
        out[c] = out[c].map(lambda x: f"{x:.2%}" if pd.notna(x) else "")
    count_cols = [
        "n_nonmatch_customers",
        "observed_bau_conversions",
        "expected_incremental_conversions",
        "expected_optimizer_conversions",
    ]
    for c in count_cols:
        out[c] = out[c].map(lambda x: f"{x:,.0f}" if pd.notna(x) else "")
    return out

plot3_executive_table = format_plot3_calculation_table(plot3_calc_summary)
display(plot3_executive_table)

# Optional drilldowns

The following cells are not required for the executive deck, but they are useful for debugging and for answering follow-up questions.

## Cell 18 — Non-match channel switch matrix

This table shows where the optimizer is moving customers from and to.

Rows = BAU channel  
Columns = optimizer channel

This helps explain *how* the optimizer changes the strategy.

In [ ]:
def channel_switch_matrix(nonmatch_table: pd.DataFrame, scenario_name: str, normalize: bool = True):
    mat = pd.crosstab(
        nonmatch_table["bau_channel"],
        nonmatch_table["optimizer_channel"],
        normalize="index" if normalize else False,
    )
    mat = mat.reindex(index=CHANNELS, columns=CHANNELS).fillna(0)
    print(scenario_name)
    display(mat)

for scenario_name, table in nonmatch_tables.items():
    channel_switch_matrix(table, scenario_name, normalize=True)

## Cell 19 — Top individual non-match examples

This displays customers with the largest predicted gain from switching away from BAU.

Use this only for internal debugging; avoid showing customer-level examples in executive material unless properly anonymized and approved.

In [ ]:
for scenario_name, table in nonmatch_tables.items():
    print(scenario_name)
    display(
        table.sort_values("delta_lift_vs_bau", ascending=False)
             .head(20)
    )

# Suggested slide storyline

Use the notebook outputs in this order:

1. **Plot 1:** BAU and optimizer allocate channels differently across PRISM propensity deciles.
2. **Plot 2:** In match cases, model scores rank observed converters toward higher lift deciles.
3. **Plot 3A:** In non-match cases, optimizer-selected channels have higher predicted lift than BAU channels.
4. **Plot 3B/3C:** Anchoring on actual BAU conversion, the model-estimated lift implies expected conversion improvement of X% to Y%, or Z% relative uplift.

Recommended caveat:

> The non-match improvement is a model-based counterfactual estimate, not yet a randomized experiment result. A live champion/challenger or holdout test would be required to validate realized lift.